# Mutual Fund Performance Analytics
## Bluestock MF Capstone - Day 4

**Author:** DEBNIL PAL  
**Date:** 2026-06-05  
**Objective:** Build a complete performance analytics engine for 40 mutual fund schemes.  
**Data Source:** SQLite Data Warehouse (`data/db/bluestock_mf.db`)  
**Benchmark:** Nifty 100  
**Risk-Free Rate:** 6.5% p.a. (India 10Y G-Sec proxy)  

---

### Metrics Computed
| # | Metric | Method |
|---|--------|--------|
| 1 | Daily Returns | `(NAV_t / NAV_{t-1}) - 1` |
| 2 | CAGR (1Y/3Y/5Y) | `(NAV_end / NAV_start)^(1/n) - 1` |
| 3 | Sharpe Ratio | `(mean(Rp) - Rf) / std(Rp) * sqrt(252)` |
| 4 | Sortino Ratio | `(mean(Rp) - Rf) / downside_std * sqrt(252)` |
| 5 | Alpha | OLS intercept * 252 |
| 6 | Beta | OLS slope vs benchmark |
| 7 | Maximum Drawdown | `min((NAV / cummax(NAV)) - 1)` |
| 8 | Tracking Error | `std(Rp - Rb) * sqrt(252)` |
| 9 | Composite Scorecard | Weighted rank-based scoring |
| 10 | Benchmark Comparison | Growth of Rs.100 |

---

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sqlalchemy import create_engine, text
from pathlib import Path
from IPython.display import display, Markdown, HTML

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# Matplotlib style
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.facecolor': '#FAFBFC',
    'axes.facecolor': '#FAFBFC',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

# Constants
RISK_FREE_RATE = 0.065
TRADING_DAYS = 252
RF_DAILY = RISK_FREE_RATE / TRADING_DAYS

print('All imports loaded successfully.')
print(f'Risk-Free Rate: {RISK_FREE_RATE*100:.1f}% p.a. ({RF_DAILY*100:.6f}% daily)')
print(f'Annualisation Factor: sqrt({TRADING_DAYS}) = {np.sqrt(TRADING_DAYS):.4f}')

## 2. Data Loading from SQLite Warehouse

In [ ]:
# Connect to SQLite warehouse
BASE_DIR = Path('.').resolve()
DB_PATH = BASE_DIR / 'data' / 'db' / 'bluestock_mf.db'
engine = create_engine(f'sqlite:///{DB_PATH}', echo=False)

def load_table(name):
    """Load a full table from SQLite."""
    df = pd.read_sql(f'SELECT * FROM {name}', engine)
    print(f'  Loaded {name:25s} -> {len(df):,} rows, {df.shape[1]} cols')
    return df

print('Loading tables from SQLite warehouse...')
print(f'Database: {DB_PATH}')
print()

In [ ]:
# Load core tables
nav_df = load_table('fact_nav')
nav_df['date'] = pd.to_datetime(nav_df['date_id'])
nav_df = nav_df.sort_values(['amfi_code', 'date']).reset_index(drop=True)

benchmark_df = load_table('fact_benchmark')
benchmark_df['date'] = pd.to_datetime(benchmark_df['date_id'])
benchmark_df = benchmark_df.sort_values(['index_name', 'date']).reset_index(drop=True)

fund_master = load_table('dim_fund')
perf_df = load_table('fact_performance')

print(f'\nTotal Funds: {fund_master["amfi_code"].nunique()}')
print(f'NAV Date Range: {nav_df["date"].min().strftime("%Y-%m-%d")} to {nav_df["date"].max().strftime("%Y-%m-%d")}')
print(f'Benchmark Indices: {benchmark_df["index_name"].unique().tolist()}')

In [ ]:
# Fund Universe Overview
display(Markdown('### Fund Universe'))
fund_overview = fund_master[['amfi_code', 'scheme_name', 'category', 'plan', 
                              'expense_ratio_pct', 'risk_category']].copy()
fund_overview.columns = ['AMFI Code', 'Scheme Name', 'Category', 'Plan', 
                          'Expense Ratio (%)', 'Risk Category']
display(fund_overview.style.set_caption('All 40 Mutual Fund Schemes'))

## 3. Daily Returns

In [ ]:
# Compute daily returns for all schemes
daily_df = nav_df.sort_values(['amfi_code', 'date']).copy()
daily_df['daily_return'] = daily_df.groupby('amfi_code')['nav'].pct_change()

print(f'Daily returns computed for {daily_df["amfi_code"].nunique()} schemes')
print(f'Total records: {len(daily_df):,}')
print(f'NaN returns (first obs per fund): {daily_df["daily_return"].isna().sum()}')
print(f'Infinite returns: {np.isinf(daily_df["daily_return"].dropna()).sum()}')

In [ ]:
# Daily return summary statistics
daily_summary = daily_df.groupby('amfi_code')['daily_return'].agg(
    ['mean', 'median', 'std', 'min', 'max', 'count']
).reset_index()
daily_summary = daily_summary.merge(fund_master[['amfi_code', 'scheme_name']], on='amfi_code')
daily_summary = daily_summary[['amfi_code', 'scheme_name', 'mean', 'median', 'std', 'min', 'max', 'count']]

display(Markdown('### Daily Return Summary Statistics'))
display(daily_summary.head(10).style.format({
    'mean': '{:.6f}', 'median': '{:.6f}', 'std': '{:.6f}',
    'min': '{:.4f}', 'max': '{:.4f}'
}).set_caption('Daily Return Statistics (First 10 Funds)'))

In [ ]:
# Daily return distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
returns = daily_df['daily_return'].dropna()
axes[0].hist(returns, bins=120, color='#1565C0', edgecolor='white', alpha=0.85)
axes[0].axvline(returns.mean(), color='#D32F2F', ls='--', lw=1.5, 
                label=f'Mean = {returns.mean():.4f}')
axes[0].axvline(returns.median(), color='#FF6F00', ls='--', lw=1.5,
                label=f'Median = {returns.median():.4f}')
axes[0].set_title('Daily Return Distribution - All Schemes', fontweight='bold')
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].legend()

# Q-Q plot
from scipy.stats import probplot
probplot(returns.values, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot (vs Normal Distribution)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nSkewness: {returns.skew():.4f}')
print(f'Kurtosis: {returns.kurtosis():.4f}')
print(f'Note: Kurtosis > 0 indicates fat tails (leptokurtic)')

## 4. CAGR Analysis (1Y / 3Y / 5Y)

In [ ]:
# Compute CAGR for all funds
max_date = nav_df['date'].max()
cagr_results = []

for code, grp in nav_df.groupby('amfi_code'):
    grp = grp.sort_values('date')
    nav_end = grp['nav'].iloc[-1]
    end_date = grp['date'].iloc[-1]
    row = {'amfi_code': code}
    
    for label, years in [('cagr_1yr', 1), ('cagr_3yr', 3), ('cagr_5yr', 5)]:
        target = end_date - pd.DateOffset(years=years)
        subset = grp[grp['date'] >= target]
        if len(subset) < 20:
            row[label] = np.nan
        else:
            nav_start = subset['nav'].iloc[0]
            actual_years = (end_date - subset['date'].iloc[0]).days / 365.25
            row[label] = (nav_end / nav_start) ** (1 / actual_years) - 1 if nav_start > 0 else np.nan
    cagr_results.append(row)

cagr_df = pd.DataFrame(cagr_results)
cagr_df = cagr_df.merge(fund_master[['amfi_code', 'scheme_name', 'category']], on='amfi_code')

display(Markdown('### CAGR Report (All Funds)'))
display(cagr_df[['scheme_name', 'category', 'cagr_1yr', 'cagr_3yr', 'cagr_5yr']]
        .sort_values('cagr_3yr', ascending=False)
        .style.format({'cagr_1yr': '{:.2%}', 'cagr_3yr': '{:.2%}', 'cagr_5yr': '{:.2%}'})
        .background_gradient(cmap='RdYlGn', subset=['cagr_1yr', 'cagr_3yr', 'cagr_5yr'])
        .set_caption('CAGR Rankings - All 40 Funds'))

In [ ]:
# Top 10 CAGR Charts
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
colors = ['#0D47A1', '#1565C0', '#1976D2', '#1E88E5', '#2196F3',
          '#42A5F5', '#64B5F6', '#90CAF9', '#BBDEFB', '#E3F2FD']

for i, (col, title) in enumerate([
    ('cagr_1yr', 'Top 10 - 1Y CAGR'),
    ('cagr_3yr', 'Top 10 - 3Y CAGR'),
    ('cagr_5yr', 'Top 10 - 5Y CAGR'),
]):
    top = cagr_df.dropna(subset=[col]).nlargest(10, col)
    top['label'] = top['scheme_name'].str[:30]
    axes[i].barh(top['label'][::-1], (top[col]*100).values[::-1],
                 color=colors[::-1], edgecolor='white', height=0.65)
    for bar in axes[i].patches:
        w = bar.get_width()
        axes[i].text(w + 0.3, bar.get_y() + bar.get_height()/2,
                     f'{w:.1f}%', va='center', fontsize=8, fontweight='bold')
    axes[i].set_title(title, fontweight='bold')
    axes[i].set_xlabel('CAGR (%)')

plt.tight_layout()
plt.show()

## 5. Sharpe Ratio Analysis

In [ ]:
# Compute Sharpe Ratio
sharpe_results = []
for code, grp in daily_df.groupby('amfi_code'):
    r = grp['daily_return'].dropna()
    if len(r) < 30:
        sharpe_results.append({'amfi_code': code, 'sharpe_ratio': np.nan})
        continue
    excess = r.mean() - RF_DAILY
    sharpe = (excess / r.std()) * np.sqrt(TRADING_DAYS)
    sharpe_results.append({'amfi_code': code, 'sharpe_ratio': round(sharpe, 4)})

sharpe_df = pd.DataFrame(sharpe_results).merge(
    fund_master[['amfi_code', 'scheme_name', 'category']], on='amfi_code'
)
sharpe_df['rank'] = sharpe_df['sharpe_ratio'].rank(ascending=False).astype(int)
sharpe_df = sharpe_df.sort_values('rank')

display(Markdown('### Sharpe Ratio Rankings'))
display(Markdown(f'**Risk-Free Rate:** {RISK_FREE_RATE*100:.1f}% p.a. | '
                 f'**Annualisation:** sqrt({TRADING_DAYS})'))
display(sharpe_df[['rank', 'scheme_name', 'category', 'sharpe_ratio']].head(15)
        .style.background_gradient(cmap='RdYlGn', subset=['sharpe_ratio'])
        .set_caption('Top 15 Funds by Sharpe Ratio'))

In [ ]:
# Top 10 Sharpe Chart
top_sharpe = sharpe_df.nlargest(10, 'sharpe_ratio').copy()
top_sharpe['label'] = top_sharpe['scheme_name'].str[:35]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top_sharpe['label'][::-1], top_sharpe['sharpe_ratio'].values[::-1],
               color=colors[::-1], edgecolor='white', height=0.65)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.02, bar.get_y() + bar.get_height()/2,
            f'{w:.2f}', va='center', fontsize=9, fontweight='bold')
ax.set_title('Top 10 Funds - Sharpe Ratio (Rf = 6.5%)', fontweight='bold', fontsize=14)
ax.set_xlabel('Sharpe Ratio')
plt.tight_layout()
plt.show()

## 6. Sortino Ratio Analysis

In [ ]:
# Compute Sortino Ratio
sortino_results = []
for code, grp in daily_df.groupby('amfi_code'):
    r = grp['daily_return'].dropna()
    downside = r[r < 0]
    if len(downside) < 10:
        sortino_results.append({'amfi_code': code, 'sortino_ratio': np.nan})
        continue
    excess = r.mean() - RF_DAILY
    sortino = (excess / downside.std()) * np.sqrt(TRADING_DAYS)
    sortino_results.append({'amfi_code': code, 'sortino_ratio': round(sortino, 4)})

sortino_df = pd.DataFrame(sortino_results).merge(
    fund_master[['amfi_code', 'scheme_name', 'category']], on='amfi_code'
)
sortino_df['rank'] = sortino_df['sortino_ratio'].rank(ascending=False).astype(int)
sortino_df = sortino_df.sort_values('rank')

display(Markdown('### Sortino Ratio Rankings'))
display(sortino_df[['rank', 'scheme_name', 'category', 'sortino_ratio']].head(15)
        .style.background_gradient(cmap='RdYlGn', subset=['sortino_ratio'])
        .set_caption('Top 15 Funds by Sortino Ratio'))

In [ ]:
# Sharpe vs Sortino comparison
merged_ratios = sharpe_df[['amfi_code', 'sharpe_ratio']].merge(
    sortino_df[['amfi_code', 'sortino_ratio']], on='amfi_code'
)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(merged_ratios['sharpe_ratio'], merged_ratios['sortino_ratio'],
           c='#1565C0', s=80, edgecolors='#333', linewidth=0.5, alpha=0.85)
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, '--', color='#999', lw=0.8, label='45-degree line')
ax.set_title('Sharpe Ratio vs Sortino Ratio', fontweight='bold', fontsize=14)
ax.set_xlabel('Sharpe Ratio')
ax.set_ylabel('Sortino Ratio')
ax.legend()
plt.tight_layout()
plt.show()

print('Sortino > Sharpe indicates asymmetric downside risk is lower than total risk.')
print('All points above the 45-degree line show funds with better downside protection.')

## 7. Alpha & Beta Analysis

In [ ]:
# Compute Alpha & Beta via OLS regression against Nifty 100
bench = benchmark_df[benchmark_df['index_name'] == 'NIFTY100'].copy()
bench = bench.sort_values('date')
bench['bench_return'] = bench['close_value'].pct_change()
bench = bench.dropna(subset=['bench_return'])[['date', 'bench_return']]

ab_results = []
for code, grp in daily_df.groupby('amfi_code'):
    merged = grp[['date', 'daily_return']].merge(bench, on='date', how='inner').dropna()
    if len(merged) < 30:
        ab_results.append({'amfi_code': code, 'beta': np.nan, 'alpha_annual': np.nan,
                           'r_squared': np.nan, 'correlation': np.nan})
        continue
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        merged['bench_return'], merged['daily_return']
    )
    ab_results.append({
        'amfi_code': code, 'beta': round(slope, 4),
        'alpha_annual': round(intercept * TRADING_DAYS, 4),
        'r_squared': round(r_value**2, 4),
        'correlation': round(r_value, 4),
    })

ab_df = pd.DataFrame(ab_results).merge(
    fund_master[['amfi_code', 'scheme_name', 'category']], on='amfi_code'
)

display(Markdown('### Alpha & Beta (vs Nifty 100)'))
display(ab_df[['scheme_name', 'category', 'beta', 'alpha_annual', 'r_squared', 'correlation']]
        .sort_values('alpha_annual', ascending=False)
        .style.format({'beta': '{:.3f}', 'alpha_annual': '{:.4f}', 
                       'r_squared': '{:.3f}', 'correlation': '{:.3f}'})
        .background_gradient(cmap='RdYlGn', subset=['alpha_annual'])
        .set_caption('Alpha & Beta Rankings'))

In [ ]:
# Alpha vs Beta Scatter
fig = px.scatter(
    ab_df.dropna(), x='beta', y=ab_df.dropna()['alpha_annual']*100,
    color='r_squared', size='r_squared',
    hover_name='scheme_name', color_continuous_scale='RdYlGn',
    title='Alpha vs Beta - All Funds (Benchmark: Nifty 100)',
    labels={'beta': 'Beta (Market Sensitivity)', 'y': 'Annualised Alpha (%)',
            'r_squared': 'R-squared'},
)
fig.add_hline(y=0, line_dash='dash', line_color='grey')
fig.add_vline(x=1.0, line_dash='dash', line_color='grey')
fig.update_layout(template='plotly_white', height=600)
fig.show()

In [ ]:
# Alpha & Beta distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vals_alpha = ab_df['alpha_annual'].dropna() * 100
axes[0].hist(vals_alpha, bins=20, color='#00695C', edgecolor='white', alpha=0.85)
axes[0].axvline(vals_alpha.mean(), color='#D32F2F', ls='--', lw=1.5,
                label=f'Mean = {vals_alpha.mean():.2f}%')
axes[0].set_title('Distribution of Annualised Alpha', fontweight='bold')
axes[0].set_xlabel('Annualised Alpha (%)')
axes[0].legend()

vals_beta = ab_df['beta'].dropna()
axes[1].hist(vals_beta, bins=20, color='#4A148C', edgecolor='white', alpha=0.85)
axes[1].axvline(1.0, color='#D32F2F', ls='--', lw=1.5, label='Market Beta = 1.0')
axes[1].axvline(vals_beta.mean(), color='#FF6F00', ls='--', lw=1.5,
                label=f'Mean = {vals_beta.mean():.2f}')
axes[1].set_title('Distribution of Beta (vs Nifty 100)', fontweight='bold')
axes[1].set_xlabel('Beta')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Maximum Drawdown Analysis

In [ ]:
# Compute Maximum Drawdown
dd_results = []
for code, grp in nav_df.groupby('amfi_code'):
    grp = grp.sort_values('date').reset_index(drop=True)
    running_max = grp['nav'].cummax()
    drawdown = (grp['nav'] / running_max) - 1
    
    idx_max_dd = drawdown.idxmin()
    max_dd = drawdown.iloc[idx_max_dd]
    peak_idx = grp['nav'][:idx_max_dd + 1].idxmax()
    dd_start = grp['date'].iloc[peak_idx]
    dd_end = grp['date'].iloc[idx_max_dd]
    
    peak_nav = grp['nav'].iloc[peak_idx]
    recovery_subset = grp[grp.index > idx_max_dd]
    recovery_rows = recovery_subset[recovery_subset['nav'] >= peak_nav]
    recovery_date = recovery_rows['date'].iloc[0].strftime('%Y-%m-%d') if len(recovery_rows) > 0 else 'Not Recovered'
    
    dd_results.append({
        'amfi_code': code,
        'max_drawdown_pct': round(max_dd * 100, 2),
        'drawdown_start': dd_start.strftime('%Y-%m-%d'),
        'drawdown_end': dd_end.strftime('%Y-%m-%d'),
        'recovery_date': recovery_date,
    })

dd_df = pd.DataFrame(dd_results).merge(
    fund_master[['amfi_code', 'scheme_name', 'category']], on='amfi_code'
)

display(Markdown('### Maximum Drawdown Analysis'))
display(dd_df[['scheme_name', 'category', 'max_drawdown_pct', 'drawdown_start', 
               'drawdown_end', 'recovery_date']]
        .sort_values('max_drawdown_pct')
        .style.background_gradient(cmap='RdYlGn', subset=['max_drawdown_pct'])
        .set_caption('Maximum Drawdown - All Funds'))

In [ ]:
# Worst 10 Drawdowns
worst10 = dd_df.nsmallest(10, 'max_drawdown_pct').copy()
worst10['label'] = worst10['scheme_name'].str[:35]

fig, ax = plt.subplots(figsize=(12, 7))
dd_colors = ['#C62828', '#D32F2F', '#E53935', '#EF5350', '#F44336',
             '#EF9A9A', '#FFCDD2', '#FF8A80', '#FF5252', '#FF1744']
bars = ax.barh(worst10['label'][::-1], worst10['max_drawdown_pct'].values[::-1],
               color=dd_colors[::-1], edgecolor='white', height=0.65)
for bar in bars:
    w = bar.get_width()
    ax.text(w - 0.5, bar.get_y() + bar.get_height()/2,
            f'{w:.1f}%', va='center', fontsize=9, fontweight='bold', color='white')
ax.set_title('Worst 10 Funds - Maximum Drawdown', fontweight='bold', fontsize=14)
ax.set_xlabel('Maximum Drawdown (%)')
plt.tight_layout()
plt.show()

## 9. Tracking Error

In [ ]:
# Compute Tracking Error vs Nifty 100
te_results = []
for code, grp in daily_df.groupby('amfi_code'):
    merged = grp[['date', 'daily_return']].merge(bench, on='date', how='inner').dropna()
    if len(merged) < 30:
        te_results.append({'amfi_code': code, 'tracking_error': np.nan})
        continue
    diff = merged['daily_return'] - merged['bench_return']
    te = diff.std() * np.sqrt(TRADING_DAYS)
    te_results.append({'amfi_code': code, 'tracking_error': round(te, 4)})

te_df = pd.DataFrame(te_results).merge(
    fund_master[['amfi_code', 'scheme_name', 'category']], on='amfi_code'
)
te_df['rank'] = te_df['tracking_error'].rank(ascending=True).astype(int)
te_df = te_df.sort_values('rank')

display(Markdown('### Tracking Error Rankings (vs Nifty 100)'))
display(te_df[['rank', 'scheme_name', 'category', 'tracking_error']].head(15)
        .style.format({'tracking_error': '{:.4f}'})
        .background_gradient(cmap='RdYlGn_r', subset=['tracking_error'])
        .set_caption('Lower TE = Closer to Benchmark'))

In [ ]:
# Tracking Error Distribution
fig, ax = plt.subplots(figsize=(10, 6))
vals_te = te_df['tracking_error'].dropna() * 100
ax.hist(vals_te, bins=20, color='#BF360C', edgecolor='white', alpha=0.85)
ax.axvline(vals_te.mean(), color='#1565C0', ls='--', lw=1.5,
           label=f'Mean = {vals_te.mean():.2f}%')
ax.set_title('Distribution of Tracking Error (vs Nifty 100)', fontweight='bold')
ax.set_xlabel('Annualised Tracking Error (%)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Composite Fund Scorecard

In [ ]:
# Build Composite Scorecard
sc = fund_master[['amfi_code', 'scheme_name', 'category', 'expense_ratio_pct']].copy()
sc = sc.merge(cagr_df[['amfi_code', 'cagr_3yr']], on='amfi_code', how='left')
sc = sc.merge(sharpe_df[['amfi_code', 'sharpe_ratio']], on='amfi_code', how='left')
sc = sc.merge(ab_df[['amfi_code', 'alpha_annual']], on='amfi_code', how='left')
sc = sc.merge(dd_df[['amfi_code', 'max_drawdown_pct']], on='amfi_code', how='left')

n = len(sc)
# Ranks
sc['cagr_rank'] = sc['cagr_3yr'].rank(ascending=False, na_option='bottom')
sc['sharpe_rank'] = sc['sharpe_ratio'].rank(ascending=False, na_option='bottom')
sc['alpha_rank'] = sc['alpha_annual'].rank(ascending=False, na_option='bottom')
sc['expense_rank'] = sc['expense_ratio_pct'].rank(ascending=True, na_option='bottom')
sc['dd_rank'] = sc['max_drawdown_pct'].rank(ascending=False, na_option='bottom')

# Normalise to 0-100
for col in ['cagr_rank', 'sharpe_rank', 'alpha_rank', 'expense_rank', 'dd_rank']:
    sc[col + '_score'] = ((n - sc[col] + 1) / n) * 100

# Weighted composite
sc['composite_score'] = (
    0.30 * sc['cagr_rank_score'] +
    0.25 * sc['sharpe_rank_score'] +
    0.20 * sc['alpha_rank_score'] +
    0.15 * sc['expense_rank_score'] +
    0.10 * sc['dd_rank_score']
).round(2)

sc['tier'] = pd.cut(sc['composite_score'], bins=[0, 60, 75, 90, 100.01],
                    labels=['Weak', 'Average', 'Strong', 'Elite'], right=False)
sc = sc.sort_values('composite_score', ascending=False).reset_index(drop=True)
sc['overall_rank'] = range(1, len(sc) + 1)

display(Markdown('### Composite Fund Scorecard'))
display(Markdown('**Weights:** 3Y CAGR (30%) | Sharpe (25%) | Alpha (20%) | '
                 'Expense Ratio (15%) | Max Drawdown (10%)'))

scorecard_display = sc[['overall_rank', 'scheme_name', 'category', 'cagr_3yr', 
                        'sharpe_ratio', 'alpha_annual', 'expense_ratio_pct',
                        'max_drawdown_pct', 'composite_score', 'tier']].copy()
scorecard_display.columns = ['Rank', 'Fund', 'Category', '3Y CAGR', 'Sharpe', 
                             'Alpha', 'Expense %', 'Max DD %', 'Score', 'Tier']

display(scorecard_display.style
        .format({'3Y CAGR': '{:.2%}', 'Sharpe': '{:.2f}', 'Alpha': '{:.4f}',
                 'Expense %': '{:.2f}', 'Max DD %': '{:.1f}', 'Score': '{:.1f}'})
        .background_gradient(cmap='RdYlGn', subset=['Score'])
        .set_caption('Complete Fund Scorecard - All 40 Funds'))

In [ ]:
# Tier distribution
tier_counts = sc['tier'].value_counts().reindex(['Elite', 'Strong', 'Average', 'Weak'])
tier_colors = {'Elite': '#1B5E20', 'Strong': '#2E7D32', 'Average': '#F57F17', 'Weak': '#C62828'}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Pie chart
axes[0].pie(tier_counts.values, labels=tier_counts.index, autopct='%1.0f%%',
            colors=[tier_colors[t] for t in tier_counts.index],
            startangle=90, textprops={'fontsize': 11})
axes[0].set_title('Fund Tier Distribution', fontweight='bold', fontsize=14)

# Top 20 scorecard bar chart
top20 = sc.head(20).copy()
top20['label'] = top20['scheme_name'].str[:35]
bar_colors = [tier_colors.get(str(t), '#999') for t in top20['tier']]
bars = axes[1].barh(top20['label'][::-1], top20['composite_score'].values[::-1],
                    color=bar_colors[::-1], edgecolor='white', height=0.65)
for bar, tier in zip(bars, top20['tier'].values[::-1]):
    w = bar.get_width()
    axes[1].text(w + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{w:.1f} [{tier}]', va='center', fontsize=7.5, fontweight='bold')
axes[1].set_title('Top 20 Funds - Composite Scorecard', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Composite Score (0-100)')
axes[1].set_xlim(0, 110)

plt.tight_layout()
plt.show()

## 11. Benchmark Comparison

In [ ]:
# Growth of Rs.100 - Top 5 Funds vs Nifty 50 & Nifty 100
start_date = max_date - pd.DateOffset(years=3)
top5_codes = sc.head(5)['amfi_code'].tolist()
top5_names = sc.head(5).set_index('amfi_code')['scheme_name'].to_dict()

fig = go.Figure()

# Fund traces
for code in top5_codes:
    sub = nav_df[(nav_df['amfi_code'] == code) & (nav_df['date'] >= start_date)].sort_values('date')
    if len(sub) == 0:
        continue
    growth = (sub['nav'] / sub['nav'].iloc[0]) * 100
    fig.add_trace(go.Scatter(x=sub['date'], y=growth, mode='lines',
                             name=top5_names[code][:40], line=dict(width=2)))

# Benchmark traces
for idx_name, label in [('NIFTY50', 'Nifty 50'), ('NIFTY100', 'Nifty 100')]:
    bsub = benchmark_df[(benchmark_df['index_name'] == idx_name) & 
                         (benchmark_df['date'] >= start_date)].sort_values('date')
    if len(bsub) == 0:
        continue
    growth = (bsub['close_value'] / bsub['close_value'].iloc[0]) * 100
    fig.add_trace(go.Scatter(x=bsub['date'], y=growth, mode='lines',
                             name=label, line=dict(width=3, dash='dash')))

fig.add_hline(y=100, line_dash='dot', line_color='grey', annotation_text='Rs.100 Investment')
fig.update_layout(
    title='Growth of Rs.100 - Top 5 Funds vs Benchmarks (Last 3 Years)',
    xaxis_title='Date', yaxis_title='Growth (Rs.)',
    template='plotly_white', height=600,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, x=0.0),
)
fig.show()

## 12. Advanced Analytics

In [ ]:
# Return vs Risk Scatter (by category)
risk_return = cagr_df[['amfi_code', 'cagr_3yr']].merge(
    daily_summary[['amfi_code', 'std']], on='amfi_code'
).merge(fund_master[['amfi_code', 'category']], on='amfi_code')

risk_return['ann_vol'] = risk_return['std'] * np.sqrt(TRADING_DAYS) * 100
risk_return['cagr_pct'] = risk_return['cagr_3yr'] * 100

fig = px.scatter(
    risk_return, x='ann_vol', y='cagr_pct', color='category',
    hover_data=['amfi_code'],
    title='Return vs Risk - 3Y CAGR vs Annualised Volatility',
    labels={'ann_vol': 'Annualised Volatility (%)', 'cagr_pct': '3-Year CAGR (%)',
            'category': 'Category'},
)
fig.update_layout(template='plotly_white', height=500)
fig.show()

In [ ]:
# CAGR Heatmap (Top 20 Funds)
heatmap_data = cagr_df.sort_values('cagr_3yr', ascending=False).head(20).copy()
heatmap_data['label'] = heatmap_data['scheme_name'].str[:30]

heatvals = heatmap_data[['cagr_1yr', 'cagr_3yr', 'cagr_5yr']].values * 100
labels = heatmap_data['label'].values

fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(heatvals, cmap='RdYlGn', aspect='auto')
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['1Y CAGR', '3Y CAGR', '5Y CAGR'])

for i in range(len(labels)):
    for j in range(3):
        val = heatvals[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.1f}%', ha='center', va='center',
                    fontsize=7.5, fontweight='bold',
                    color='white' if abs(val) > 15 else 'black')

cbar = fig.colorbar(im, ax=ax, shrink=0.6)
cbar.set_label('CAGR (%)')
ax.set_title('CAGR Comparison Heatmap - Top 20 Funds', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Risk Radar - Top 5 Funds
top5_radar = sc.head(5).copy()
categories_radar = ['3Y CAGR', 'Sharpe', 'Alpha', 'Low Expense', 'Low Drawdown']
score_cols = ['cagr_rank_score', 'sharpe_rank_score', 'alpha_rank_score',
              'expense_rank_score', 'dd_rank_score']

fig = go.Figure()
for i, (_, row) in enumerate(top5_radar.iterrows()):
    values = [row[c] for c in score_cols]
    values.append(values[0])  # close the radar
    fig.add_trace(go.Scatterpolar(
        r=values, theta=categories_radar + [categories_radar[0]],
        fill='toself', name=row['scheme_name'][:35],
        opacity=0.7
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title='Risk-Adjusted Performance Radar - Top 5 Funds',
    template='plotly_white', height=600,
)
fig.show()

In [ ]:
# Category Performance Box Plot
cat_perf = cagr_df[['amfi_code', 'cagr_3yr']].merge(
    fund_master[['amfi_code', 'category']], on='amfi_code'
)
cat_perf['cagr_3yr_pct'] = cat_perf['cagr_3yr'] * 100

fig = px.box(cat_perf, x='category', y='cagr_3yr_pct', color='category',
             title='3-Year CAGR Distribution by Category',
             labels={'cagr_3yr_pct': '3Y CAGR (%)', 'category': 'Category'})
fig.update_layout(template='plotly_white', height=500, showlegend=False)
fig.show()

In [ ]:
# Rolling 1-Year Sharpe - Top 5 Funds
fig, ax = plt.subplots(figsize=(14, 7))
roll_colors = ['#0D47A1', '#1B5E20', '#E65100', '#4A148C', '#BF360C']

for i, (_, row) in enumerate(sc.head(5).iterrows()):
    sub = daily_df[daily_df['amfi_code'] == row['amfi_code']].sort_values('date')
    r = sub.set_index('date')['daily_return']
    rolling_excess = r.rolling(TRADING_DAYS).mean() - RF_DAILY
    rolling_std = r.rolling(TRADING_DAYS).std()
    rolling_sharpe = (rolling_excess / rolling_std) * np.sqrt(TRADING_DAYS)
    ax.plot(rolling_sharpe.index, rolling_sharpe.values,
            label=row['scheme_name'][:30], color=roll_colors[i], lw=1.5)

ax.axhline(0, color='#999', ls='--', lw=0.8)
ax.set_title('Rolling 1-Year Sharpe Ratio - Top 5 Funds', fontweight='bold', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Rolling Sharpe Ratio')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Expense Ratio vs 3Y CAGR
exp_return = cagr_df[['amfi_code', 'cagr_3yr']].merge(
    fund_master[['amfi_code', 'expense_ratio_pct', 'plan']], on='amfi_code'
)
exp_return['cagr_3yr_pct'] = exp_return['cagr_3yr'] * 100

fig = px.scatter(exp_return, x='expense_ratio_pct', y='cagr_3yr_pct', 
                 color='plan', symbol='plan',
                 title='Expense Ratio vs 3Y CAGR - Direct vs Regular Plans',
                 labels={'expense_ratio_pct': 'Expense Ratio (%)',
                         'cagr_3yr_pct': '3Y CAGR (%)'})
fig.update_layout(template='plotly_white', height=500)
fig.show()

print('Direct plans consistently show lower expense ratios and can deliver higher net returns.')

## 12. Key Findings

In [ ]:
# Key Findings
top1 = sc.iloc[0]
top_sharpe = sharpe_df.iloc[0]
top_sortino = sortino_df.iloc[0]
top_cagr = cagr_df.sort_values('cagr_3yr', ascending=False).iloc[0]
worst_dd = dd_df.sort_values('max_drawdown_pct').iloc[0]
best_alpha = ab_df.sort_values('alpha_annual', ascending=False).iloc[0]

findings = f"""
## Key Performance Findings

### 1. Best Overall Fund
**{top1['scheme_name']}** with a composite score of **{top1['composite_score']:.1f}/100** ({top1['tier']})

### 2. Highest Risk-Adjusted Returns (Sharpe)
**{top_sharpe['scheme_name']}** - Sharpe Ratio: **{top_sharpe['sharpe_ratio']:.2f}**

### 3. Best Downside Protection (Sortino)
**{top_sortino['scheme_name']}** - Sortino Ratio: **{top_sortino['sortino_ratio']:.2f}**

### 4. Strongest 3-Year Growth
**{top_cagr['scheme_name']}** - 3Y CAGR: **{top_cagr['cagr_3yr']*100:.2f}%**

### 5. Highest Alpha Generator
**{best_alpha['scheme_name']}** - Alpha: **{best_alpha['alpha_annual']*100:.2f}%** annualised

### 6. Deepest Drawdown
**{worst_dd['scheme_name']}** - Max Drawdown: **{worst_dd['max_drawdown_pct']:.2f}%**

### 7. Fund Tier Distribution
- Elite: {(sc['tier']=='Elite').sum()} funds
- Strong: {(sc['tier']=='Strong').sum()} funds
- Average: {(sc['tier']=='Average').sum()} funds
- Weak: {(sc['tier']=='Weak').sum()} funds

### 8. Average Market Sensitivity
Average Beta = **{ab_df['beta'].dropna().mean():.2f}** (vs Nifty 100)

### 9. Average Risk-Adjusted Performance
Average Sharpe = **{sharpe_df['sharpe_ratio'].mean():.2f}** across all 40 funds

### 10. Top 5 Recommended Funds
"""

for i, (_, row) in enumerate(sc.head(5).iterrows()):
    findings += f"\n{i+1}. **{row['scheme_name']}** - Score: {row['composite_score']:.1f} | Tier: {row['tier']}\n"

display(Markdown(findings))

## 13. Conclusion

### Summary
This analysis evaluated **40 mutual fund schemes** across **10 performance metrics** using the SQLite data warehouse as the primary data source.

### Methodology
| Parameter | Value |
|-----------|-------|
| NAV Source | SQLite warehouse (fact_nav) |
| Benchmark | Nifty 100 (fact_benchmark) |
| Risk-Free Rate | 6.50% p.a. |
| Annualisation Factor | sqrt(252) |
| CAGR Periods | 1Y, 3Y, 5Y |
| Scorecard Weights | CAGR 30%, Sharpe 25%, Alpha 20%, Expense 15%, DD 10% |
| Regression Method | OLS (scipy.stats.linregress) |

### Deliverables
- 8 CSV exports in `data/processed/`
- 19 professional charts in `reports/charts/`
- Performance summary report in `reports/performance_summary.md`
- Validation report with ALL PASS status

### Disclaimer
Past performance is not indicative of future results. This analysis is for educational and research purposes only. Mutual fund investments are subject to market risks.

---

*Report prepared by DEBNIL PAL | Bluestock MF Capstone Project | Day 4*

In [ ]:
# Validation Summary
val_path = Path('data/processed/day4_validation_report.csv')
if val_path.exists():
    val_df = pd.read_csv(val_path)
    display(Markdown('### Day 4 Validation Report'))
    display(val_df.style.applymap(
        lambda v: 'background-color: #C8E6C9' if v == 'PASS' else 
                  ('background-color: #FFCDD2' if v == 'FAIL' else ''),
        subset=['status']
    ).set_caption('Validation Status'))
else:
    print('Run the performance_analytics.py pipeline first to generate validation data.')

print('\nDay 4 Performance Analytics notebook complete.')